<a href="https://colab.research.google.com/github/elite23406951-create/google.colab/blob/main/%E3%80%8CZ_Image_Turbo_jupyter_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
!git clone https://github.com/comfyanonymous/ComfyUI

%cd /content/ComfyUI
!pip install -r requirements.txt

!apt -y install -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors -d /content/ComfyUI/models/diffusion_models -o z-image-turbo-fp8-e4m3fn.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors -d /content/ComfyUI/models/clip -o qwen_3_4b.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors -d /content/ComfyUI/models/vae -o ae.safetensors

Cloning into 'ComfyUI'...
remote: Enumerating objects: 47721, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 47721 (delta 83), reused 65 (delta 57), pack-reused 47598 (from 3)
Receiving objects: 100% (47721/47721), 93.60 MiB | 13.90 MiB/s, done.
Resolving deltas: 100% (32286/32286), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.3/342.3 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 MB 11.9 MB/s eta

In [2]:
# @title
%cd /content/ComfyUI

import os, random, time

import torch
import numpy as np
from PIL import Image

from nodes import NODE_CLASS_MAPPINGS

UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

# Google Cloud Storage 設定
!pip install google-cloud-storage -q

from google.cloud import storage
import json

# 初始化 GCS 客戶端
BUCKET_NAME = "your-bucket-name"  # 替換為您的 bucket 名稱
storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

def upload_image_to_gcs(local_path, destination_blob_name):
    """上傳圖片到 Google Cloud Storage"""
    try:
        blob = bucket.blob(destination_blob_name)
        blob.upload_from_filename(local_path)
        gcs_path = f"gs://{BUCKET_NAME}/{destination_blob_name}"
        print(f"✓ 已成功上傳至: {gcs_path}")
        return gcs_path
    except Exception as e:
        print(f"✗ 上傳失敗: {str(e)}")
        return None

with torch.inference_mode():
    unet = UNETLoader.load_unet("z-image-turbo-fp8-e4m3fn.safetensors", "fp8_e4m3fn_fast")[0]
    clip = CLIPLoader.load_clip("qwen_3_4b.safetensors", type="lumina2")[0]
    vae = VAELoader.load_vae("ae.safetensors")[0]

@torch.inference_mode()
def generate(input):
    tmp_dir="/content/ComfyUI/output"
    os.makedirs(tmp_dir, exist_ok=True)

    values = input["input"]

    positive_prompt = values['positive_prompt']
    negative_prompt = values['negative_prompt']
    seed = values['seed'] # 0
    steps = values['steps'] # 9
    cfg = values['cfg'] # 1.0
    sampler_name = values['sampler_name'] # euler
    scheduler = values['scheduler'] # simple
    denoise = values['denoise'] # 1.0
    width = values['width'] # 1024
    height = values['height'] # 1024
    batch_size = values['batch_size'] # 1.0

    if seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)

    positive = CLIPTextEncode.encode(clip, positive_prompt)[0]
    negative = CLIPTextEncode.encode(clip, negative_prompt)[0]
    latent_image = EmptyLatentImage.generate(width, height, batch_size=batch_size)[0]
    samples = KSampler.sample(unet, seed, steps, cfg, sampler_name, scheduler, positive, negative, latent_image, denoise=denoise)[0]
    decoded = VAEDecode.decode(vae, samples)[0].detach()

    # 生成圖片檔名
    timestamp = int(time.time())
    image_filename = f"z_image_turbo_{timestamp}.png"
    local_path = f"{tmp_dir}/{image_filename}"

    # 保存圖片到本地
    Image.fromarray(np.array(decoded*255, dtype=np.uint8)[0]).save(local_path)

    # 上傳到 Google Cloud Storage
    gcs_path = upload_image_to_gcs(local_path, f"generated-images/{image_filename}")

    # 返回 GCS 路徑或本地路徑
    result = {
        "local_path": local_path,
        "gcs_path": gcs_path if gcs_path else None
    }

    return local_path  # 返回本地路徑以便在 Colab 中預覽

/content/ComfyUI


WARNING WARNING WARNING
If you are on nvidia 20 series and above it is required that you update your pytorch to cu130 or higher.



In [ ]:
input = {
    "input": {
        "positive_prompt": "cute young East Asian woman with extreme baby face, very round and soft facial structure, heavily chubby apple cheeks, short lower face proportion, soft tiny chin, oversized large round dark eyes with double eyelids and long curled lashes, sparkling clear eyes with soft catchlights, very small short button nose, tiny plump soft pink lips, fair luminous skin with strong natural rosy blush, poreless smooth skin, high rounded forehead, pure innocent doll-like features, exaggerated cute proportions, highly detailed face, soft natural lighting, Sailor uniform, pleated skirt。",
        "negative_prompt": "blurry ugly bad",
        "width": 1024,
        "height": 1024,
        "batch_size": 1,
        "seed": 0,
        "steps": 9,
        "cfg": 1,
        "sampler_name": "euler",
        "scheduler": "simple",
        "denoise": 1.0,
    }
}

output = generate(input)
Image.open(output)

  0%|          | 0/9 [00:00<?, ?it/s]